# PCA step by step

40 points with 3 columns and 2 classes, reduced to 2 columns by hand, then checked against scikit-learn.

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

## The data

Two clouds of 20 points each, drawn from 3-D normal distributions centred at (0, 0, 0) and (1, 1, 1).

In [ ]:
np.random.seed(23)
class1 = np.random.multivariate_normal([0, 0, 0], np.eye(3), 20)
class2 = np.random.multivariate_normal([1, 1, 1], np.eye(3), 20)
cols = ["feature1", "feature2", "feature3"]
df = pd.concat([pd.DataFrame(class1, columns=cols).assign(target=1),
                pd.DataFrame(class2, columns=cols).assign(target=0)], ignore_index=True)
df.head()

In [ ]:
px.scatter_3d(df, x="feature1", y="feature2", z="feature3", color=df["target"].astype(str))

## Step 1: mean centring (here with StandardScaler, which also scales each column)

In [ ]:
X = StandardScaler().fit_transform(df[cols])
X.mean(axis=0).round(6)   # every column now has mean 0

## Step 2: the covariance matrix

In [ ]:
C = np.cov(X.T)            # np.cov wants one row per variable, so we transpose
C.round(3)                 # diagonal: variances; off-diagonal: covariances; symmetric

## Step 3: eigenvalues and eigenvectors

In [ ]:
# eigh is made for symmetric matrices like C; it returns eigenvalues in ascending order
values, vectors = np.linalg.eigh(C)
# sort from largest to smallest eigenvalue
order = np.argsort(values)[::-1]
values, vectors = values[order], vectors[:, order]
print(values.round(3))
print(vectors.round(3))    # each COLUMN is one eigenvector

## Step 4: keep the top 2 eigenvectors

In [ ]:
W = vectors[:, :2].T       # 2 rows, one per principal component, each with 3 numbers
W.round(3)

A common mistake: `vectors[0:2]` takes the first two **rows**, but the eigenvectors are the **columns**. The rows are not principal components, and the variance along them is not the largest:

In [ ]:
wrong = X @ vectors[0:2].T
right = X @ W.T
print("variance along the wrong axes:", wrong.var(axis=0, ddof=1).round(3))
print("variance along PC1, PC2:      ", right.var(axis=0, ddof=1).round(3))

## Step 5: project the data

In [ ]:
Z = X @ W.T                # (40 x 3) @ (3 x 2) = 40 x 2
new_df = pd.DataFrame(Z, columns=["PC1", "PC2"])
new_df["target"] = df["target"].astype(str)
px.scatter(new_df, x="PC1", y="PC2", color="target")

## Check against scikit-learn

In [ ]:
pca = PCA(n_components=2)
Z_sk = pca.fit_transform(X)
print("eigenvalues   :", values[:2].round(3))
print("sklearn       :", pca.explained_variance_.round(3))
print("variance share:", pca.explained_variance_ratio_.round(3))
# same columns up to sign: an eigenvector and its negative point along the same line
print(np.allclose(np.abs(Z), np.abs(Z_sk)))